In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datetime import datetime

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-crunch-round-1/sample_submission.csv
/kaggle/input/data-crunch-round-1/train.csv
/kaggle/input/data-crunch-round-1/test.csv


In [3]:
df = pd.read_csv("/kaggle/input/data-crunch-round-1/train.csv")

def to_date(year, month, day):
    return datetime(year=2018+year, month=month, day=day)

def from_date(date):
    return date.year-2018, date.month, date.day

df['Date'] = df.apply(lambda row: datetime(year=2018+row['Year'], 
                                                     month=row['Month'], 
                                                     day=row['Day']), axis=1)
df['Date'] = df.apply(lambda row: to_date(row['Year'], row['Month'], row['Day']), axis=1)
df.set_index('Date', inplace=True)
target = ['Avg_Temperature', 'Radiation', 'Rain_Amount','Wind_Speed', 'Wind_Direction']
df = df[['kingdom'] + target]
kingdoms = [df[df['kingdom']==k] for k in df['kingdom'].unique()]


def predict(data, n, th=100):
    N = len(data)
    fft_values = np.fft.fft(data)
    freqs = np.fft.fftfreq(N, d=1)
    extended_indexes = np.arange(N+1, N+1+n)

    reconstructed_fet = np.zeros(len(extended_indexes), dtype=np.complex128)
    
    threshold = th
    for i, freq in enumerate(freqs[:N//2]): 
        if abs(freq) > threshold:
            continue
        amplitude = np.abs(fft_values[i]) / N
        phase = np.angle(fft_values[i])
        reconstructed_fet += amplitude * np.exp(1j * (2 * np.pi * freq * extended_indexes + phase))

    reconstructed_fet = np.real(reconstructed_fet)
    return reconstructed_fet


def predict_many(data, n, th=0.4):
    return np.array([predict(fet, n, th) for fet in data])

def create_dataframe(indexes, data, columns, default=dict()):
    df = pd.DataFrame(data, index=indexes, columns=columns)
    for key, val in default.items():
        df.insert(0, key, val)
    return df

def create_next_date_window_n(start, n):
    dates = np.full(n, start, dtype='datetime64[ns]')
    indexes = np.arange(1, n+1)
    timedeltas = np.array(indexes, dtype='timedelta64[D]')
    return dates + timedeltas

def create_next_date_window_stop(start, end):
    n = (end-start).days
    return create_next_date_window_n(start, n)

In [13]:
def analyze_periods(data):
    N = len(data)
    fft_values = np.fft.fft(data)
    freqs = np.fft.fftfreq(N, d=1)
    
    magnitudes = np.abs(fft_values[:N//2]) / N
    sorted_indices = np.argsort(magnitudes)[::-1]
    
    periods = [1 / freqs[i] if freqs[i] != 0 else np.inf for i in sorted_indices]
    
    return periods


In [15]:
df[df['kingdom']=='Arcadia'][target].to_numpy().T[0]

array([25.5, 25.8, 26. , ..., 25. , 24. , 23.8])

In [17]:
df[df['kingdom']=='Arcadia'][target].to_numpy().T[0][-10:]

array([24.2, 24.3, 24.6, 23.1, 23.9, 24.8, 25.2, 25. , 24. , 23.8])

In [18]:
analyze_periods(df[df['kingdom']=='Arcadia'][target].to_numpy().T[0])

[inf,
 354.0,
 177.0,
 404.5714285714286,
 1416.0,
 188.8,
 236.0,
 566.4,
 708.0,
 283.2,
 472.0,
 118.0,
 123.1304347826087,
 2832.0,
 944.0,
 101.14285714285715,
 166.58823529411765,
 33.31764705882353,
 57.795918367346935,
 62.93333333333334,
 217.84615384615384,
 91.35483870967742,
 257.4545454545455,
 70.8,
 37.760000000000005,
 69.07317073170732,
 202.2857142857143,
 53.43396226415094,
 314.66666666666663,
 48.0,
 134.85714285714286,
 51.490909090909085,
 19.943661971830984,
 17.26829268292683,
 141.6,
 49.684210526315795,
 28.039603960396043,
 157.33333333333331,
 48.827586206896555,
 15.063829787234043,
 19.006711409395972,
 13.615384615384615,
 38.79452054794521,
 38.270270270270274,
 42.90909090909091,
 88.5,
 12.990825688073395,
 64.36363636363637,
 17.163636363636364,
 17.06024096385542,
 37.26315789473684,
 113.28,
 35.848101265822784,
 20.085106382978722,
 11.465587044534415,
 12.756756756756756,
 19.804195804195803,
 149.05263157894737,
 17.48148148148148,
 61.565217391

In [6]:
def analyze_periods(data, n, th=100):
    N = len(data)
    fft_values = np.fft.fft(data)
    freqs = np.fft.fftfreq(N, d=1)
    
    # Compute magnitudes and filter frequencies within threshold
    magnitudes = np.abs(fft_values[:N//2]) / N
    valid_indices = np.where(np.abs(freqs[:N//2]) <= th)[0]
    
    # Sort frequencies by magnitude in descending order
    sorted_indices = valid_indices[np.argsort(magnitudes[valid_indices])[::-1]]
    
    # Convert frequencies to periods (T = 1/f) while avoiding division by zero
    periods = [1 / freqs[i] if freqs[i] != 0 else np.inf for i in sorted_indices]
    
    return periods

In [7]:
k

,kingdom,Avg_Temperature,Radiation,Rain_Amount,Wind_Speed,Wind_Direction
Date,,,,,,
2019-04-01,Winterfell,23.0,21.87,11.31,10.5,167
2019-04-02,Winterfell,23.0,24.37,10.40,9.9,227
2019-04-03,Winterfell,23.3,20.23,22.23,4.8,354
2019-04-04,Winterfell,24.3,24.84,0.65,5.7,308
2019-04-05,Winterfell,24.3,24.44,1.17,8.0,182
...,...,...,...,...,...,...
2026-12-27,Winterfell,22.4,22.18,0.26,8.7,142
2026-12-28,Winterfell,21.9,18.89,3.51,10.5,81
2026-12-29,Winterfell,21.8,23.23,0.00,12.6,76


In [4]:
preds = list()

end_date = to_date(9, 5, 31)


for k in kingdoms:
    n_preds = (end_date-k.index[-1]).days
    name = k['kingdom'].unique()[0]
    print(name)
    k_fet = k[target].to_numpy().T
    k_pred = predict_many(k[target].to_numpy().T, n_preds).T
    dw = create_next_date_window_n(k.index[-1], n_preds)
    pred_df = create_dataframe(dw, k_pred, target, {'kingdom': name})
    preds.append(pred_df)
df_pred = pd.concat(preds, axis=0)

Arcadia
Atlantis
Avalon
Camelot
Dorne
Eden
El Dorado
Elysium
Emerald City
Helios
Krypton
Metropolis
Midgar
Midgard
Mordor
Neo-City
Neo-Tokyo
Nirvana
Olympus
Pandora
Rapture
Rivendell
Serenity
Shangri-La
Solara
Solstice
Sunspear
Utopia
Valyria
Winterfell


In [5]:
def get_pred(year, month, day, kingdom):
    return df_pred[(df_pred.index == to_date(year, month, day)) & (df_pred['kingdom']==kingdom)]

In [31]:
train_df = pd.read_csv('/kaggle/input/data-crunch-round-1/train.csv')
test_df = pd.read_csv('/kaggle/input/data-crunch-round-1/test.csv')
id_cols = ['ID', 'Year', 'Month', 'Day', 'kingdom']

id_df = pd.concat([train_df[id_cols], test_df[id_cols]], axis=0)
id_df.set_index('ID', inplace=True)

def pred_from_id(id):
    r = id_df[id_df.index==id]
    year, month, day, kingdom = r.iloc[0].to_numpy()
    return get_pred(year, month, day, kingdom)[target].to_numpy()[0]

In [33]:
def pred_from_ids(ids):
    final_pred = list(map(pred_from_id, ids))
    final_pred_df = create_dataframe(ids, final_pred, target)
    final_pred_df = final_pred_df.reset_index().rename(columns={'index': 'ID'})
    final_pred_df['Avg_Temperature'] = final_pred_df['Avg_Temperature'].apply(lambda x: x - 273.15 if x > 200 else x)
    return final_pred_df

In [45]:
ids = pd.read_csv("/kaggle/input/data-crunch-round-1/train.csv")['ID'].to_numpy()
ids[:100]

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100])

In [35]:
pred_df

,ID,Avg_Temperature,Radiation,Rain_Amount,Wind_Speed,Wind_Direction
0,84961,25.787852,22.020727,16.025933,11.832419,247.436357
1,84962,27.405727,23.293370,8.069073,14.470094,213.731812
2,84963,26.915160,23.293370,8.069073,14.470094,213.731812
3,84964,23.832505,21.362747,31.359934,9.883312,153.919217
4,84965,28.027483,22.103871,2.902246,19.867404,227.368844
...,...,...,...,...,...,...
4525,89486,28.676827,23.623594,2.216718,23.936603,237.986883
4526,89487,28.376126,23.806337,2.814234,31.140017,224.492631
4527,89488,26.750045,19.997153,7.400376,16.412326,232.541027
4528,89489,28.571267,22.786278,3.103946,16.275112,184.610298


In [40]:
ids = train_df['ID'].to_numpy()
pred_df = pred_from_ids(ids)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [5]:
df = pd.read_csv("/kaggle/input/data-crunch-round-1/train.csv")
df['Date'] = df.apply(lambda row: datetime(year=2018+row['Year'], 
                                                     month=row['Month'], 
                                                     day=row['Day']), axis=1)
df

,ID,Year,Month,Day,kingdom,latitude,longitude,Avg_Temperature,Avg_Feels_Like_Temperature,Temperature_Range,Feels_Like_Temperature_Range,Radiation,Rain_Amount,Rain_Duration,Wind_Speed,Wind_Direction,Evapotranspiration,Date
0,1,1,4,1,Arcadia,24.280002,-37.229980,25.50,30.50,8.5,10.3,22.52,58.89,16,8.6,283,1.648659,2019-04-01
1,2,1,4,1,Atlantis,22.979999,-37.329990,299.65,305.15,5.9,8.2,22.73,11.83,12,15.8,161,1.583094,2019-04-01
2,3,1,4,1,Avalon,22.880000,-37.130006,26.30,31.50,5.2,6.4,22.73,11.83,12,15.8,161,1.593309,2019-04-01
3,4,1,4,1,Camelot,24.180003,-36.929994,24.00,28.40,8.2,10.7,22.67,75.27,16,6.4,346,1.638997,2019-04-01
4,5,1,4,1,Dorne,25.780002,-37.530000,28.00,32.80,5.7,10.2,22.35,4.81,8,16.7,185,1.719189,2019-04-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84955,84956,8,12,31,Solstice,25.479998,-36.329990,25.60,28.60,3.4,3.5,19.41,0.13,1,14.8,90,1.562346,2026-12-31
84956,84957,8,12,31,Sunspear,26.580005,-37.530000,25.80,28.90,2.8,3.7,20.98,0.26,2,16.3,91,1.607436,2026-12-31
84957,84958,8,12,31,Utopia,23.979999,-37.630006,298.75,301.65,7.6,9.2,22.67,0.00,0,12.6,71,1.710188,2026-12-31
84958,84959,8,12,31,Valyria,24.280002,-35.729980,25.60,28.10,4.0,3.8,19.72,0.00,0,16.3,54,1.613430,2026-12-31
